# Zero-Shot Classification

In [ ]:
# HuggingFace Login
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HUGGING_FACE_TOKEN")
login(hf_token)

from transformers import logging
logging.set_verbosity_error()

In [ ]:
import polars as pl
import os
import numpy as np
import random
import torch
import pandas as pd
import gc
import csv
from transformers import pipeline

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else -1
FILE_PATH = 'getText/datasetsPrep'
OUTPUT_DIR = 'zeroShotOutputs'
BATCH_SIZE = 4

In [ ]:
ZERO_SHOT_MODELS = [
    "cross-encoder/nli-deberta-v3-small", # low capacity
    # "cross-encoder/nli-MiniLM2-L6-H768",
    # "MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary",
    # "MoritzLaurer/roberta-base-zeroshot-v2.0-c", # not a large amount of improvement after adding these models so excluding them
    "typeform/distilbert-base-uncased-mnli", # medium capacity
    "valhalla/distilbart-mnli-12-3", # higher capacity
]

# Exact mapping taken from https://www.uni-bamberg.de/fileadmin/eng-ling/fs/Chapter_21/23DimensionsofEnglish.html which has extracted the same from Biber and Conrad's Variation in English (https://doi.org/10.4324/9781315840888)
BIBER_LABEL_MAP = {
    "factor_1": {
        "informational, dense, precise": -1,
        "involved, interactive, affective": 1
    },
    "factor_2": {
        "non-narrative, expository, informational": -1,
        "narrative, event-focused, storytelling": 1
    },
    "factor_3": {
        "situation-dependent, context-bound, implicit": -1,
        "explicit, context-independent, elaborated": 1
    },
    "factor_4": {
        "non-persuasive, non-argumentative, neutral": -1,
        "persuasive, argumentative, modalized": 1
    },
    "factor_5": {
        "non-abstract, concrete, human-centered": -1,
        "abstract, impersonal, technical": 1
    },
    "factor_6": {
        "compressed, dense, clause-poor": -1,
        "elaborated, expanded, clause-rich": 1
    }
}

# Using different prompt templates increases robustness. Ability to modify for different factors. 
TEMPLATES = ["This example is {}.", "This text is {}.", "This text is written in a {} style.", "The writing style of this text is {}.", "This text shows {} characteristics."]

default_template = "This text is {}."


In [ ]:
# Flatten Biber label map.
all_labels = []
label_to_factor = {}

for factor, description in BIBER_LABEL_MAP.items():
    for label in description.keys():
        all_labels.append(label)
        label_to_factor[label] = factor



In [ ]:
label_to_factor

In [ ]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    pl.set_random_seed(random_state)
    pd.core.common.random_state(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state

RANDOM_STATE = set_random_states(1618)

In [ ]:
def make_dfs(df, output_dir, output_file_name):
    os.makedirs(f"./{output_dir}/", exist_ok=True)
    # Save CSV.
    df.to_csv(f"./{output_dir}/{output_file_name}.csv", index=False, lineterminator="\n", quoting=csv.QUOTE_ALL)

    # Save JSON
    df.to_json(f"./{output_dir}/{output_file_name}.json", orient="records", indent=2)

In [ ]:
# Load data.
for folder in os.listdir(f'./{FILE_PATH}'):
    if os.path.isdir(f'./{FILE_PATH}/{folder}'):
        for file in os.listdir(f'./{FILE_PATH}/{folder}'):
            if file.endswith(".csv"):
                temp_file_path = f'./{FILE_PATH}/{folder}/{file}'
                temp_tag = file.replace('_train.csv', '')
                temp_df = pl.read_csv(temp_file_path)
                temp_df = (
                    temp_df
                    .with_row_index("index_num") # , offset=1) if you want to start index from 1
                    .with_columns(
                        (pl.lit(temp_tag) + "_" + pl.col("index_num").cast(pl.Utf8)).alias("doc_id")
                    )).select(['text', 'doc_id'])
                
                # Remove invalid data.
                temp_df = temp_df.with_columns(
                    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
                )
                print(temp_df.group_by("tag").len())
                print(f"Total Texts Before Empty String Removal: {len(temp_df)}")

                temp_df = temp_df.with_columns(pl.col("text").str.strip_chars().alias("text")).filter(pl.col("text").is_not_null() & (pl.col("text") != ""))

                temp_df = temp_df.with_columns(
                    pl.col("doc_id").str.split("_").list.get(0).alias("tag")
                )
                print(temp_df.group_by("tag").len())
                print(f"Total Texts After Empty String Removal: {len(temp_df)}")

                # Set up df for use.
                df = pl.DataFrame({
                    "doc_id": temp_df['doc_id'].to_list(),
                    "text": temp_df['text'].to_list()
                })

                # Light preprocessing to strip extra whitespace.
                df = df.with_columns(
                    pl.col("text")
                    .str.strip_chars()
                    .str.replace_all(r"\s+", " ")
                    .str.replace_all(r"^\s*-\s*", "") # Remove dashes at the beginning of texts.
                    .str.replace_all(r"^\s*\d+\.\s*", "") # Remove numbers in 1., 2., 3. format at the beginning of the text. 
                )
                temp_df = temp_df.to_pandas()
                texts = temp_df['text'].values.tolist()
                doc_ids = temp_df['doc_id'].values.tolist()

                assert len(texts) == len(doc_ids), "texts and doc_ids are not of the same length."
                print(f"Using {len(texts)} train datapoints...")
                all_dfs = []
                for model_name in ZERO_SHOT_MODELS:
                    classifier = pipeline(
                        "zero-shot-classification",
                        model=model_name,
                        device=DEVICE
                    )

                    # Initialize factor storage
                    temp_factors_list = {
                        'model_name': [model_name] * len(doc_ids),
                        'doc_id': doc_ids,
                        **{factor: [] for factor in BIBER_LABEL_MAP}
                    }

                    # Dictionary to accumulate scores per factor across templates
                    factor_scores_accum = {factor: [0.0] * len(texts) for factor in BIBER_LABEL_MAP}

                    # Loop over all templates
                    for template in TEMPLATES:

                        with torch.no_grad():
                            outputs = classifier(
                                texts,
                                candidate_labels=all_labels,
                                hypothesis_template=template,
                                multi_label=True,
                                batch_size = BATCH_SIZE
                            )

                            if isinstance(outputs, dict):
                                outputs = [outputs]

                            # Accumulate weighted scores per factor
                            for j, output in enumerate(outputs):
                                for label, score in zip(output['labels'], output['scores']):
                                    f = label_to_factor[label]
                                    weight = BIBER_LABEL_MAP[f][label]
                                    factor_scores_accum[f][j] += weight * score

                    # Average over templates
                    n_templates = len(TEMPLATES)
                    for factor in BIBER_LABEL_MAP:
                        factor_scores_accum[factor] = [s / n_templates for s in factor_scores_accum[factor]]
                        temp_factors_list[factor].extend(factor_scores_accum[factor])

                    # Convert to DataFrame and append to all_dfs
                    all_dfs.append(pd.DataFrame(temp_factors_list))

                    # Free memory
                    del classifier
                    torch.cuda.empty_cache()
                    gc.collect()

                df = pd.concat(all_dfs)
                # Simple averaging will prevent over-confidence. 
                mean_scores = df.drop(columns='model_name').groupby("doc_id").mean().reset_index()

                make_dfs(df, f"{OUTPUT_DIR}/{folder}", f"{file.replace('.csv', '')}AllModels")
                make_dfs(mean_scores, f"{OUTPUT_DIR}/{folder}", f"{file.replace('.csv', '')}MeanScores")

                print(pl.from_pandas(df))
                print(pl.from_pandas(mean_scores))

